# 💀 Day 04a: Deadlocks & Recovery — Detection, Prevention, WAL Recovery

---

## 🎯 What You'll Master Today
- Deadlock detection (wait-for graph)
- Deadlock prevention (wait-die, wound-wait)
- Recovery techniques: undo, redo, ARIES
- Checkpointing for faster recovery

---

```
╔══════════════════════════════════════════════════════════════════╗
║                  DEADLOCK IN DATABASES                           ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  T1: LOCK(A) ────→ wants LOCK(B)                                ║
║        ↑                   │                                     ║
║        │               WAIT ↓                                    ║
║     WAIT ↑                 │                                     ║
║        │            ←──── T2: LOCK(B) ────→ wants LOCK(A)       ║
║                                                                  ║
║  T1 waits for T2.  T2 waits for T1.  → DEADLOCK!               ║
║                                                                  ║
║  Solutions:                                                      ║
║  1. Detection: find cycles in wait-for graph → abort victim     ║
║  2. Prevention: timestamp-based protocols (no cycles possible)  ║
║  3. Timeout: abort if waiting too long                           ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Wait-For Graph — Deadlock Detection
# ============================================

from collections import defaultdict

class WaitForGraph:
    def __init__(self):
        self.graph = defaultdict(set)  # txn → set of txns it waits for
    
    def add_wait(self, waiter, holder):
        self.graph[waiter].add(holder)
        print(f"  {waiter} → waits for → {holder}")
    
    def detect_cycle(self):
        """DFS-based cycle detection."""
        visited = set()
        rec_stack = set()
        cycle_path = []
        
        def dfs(node, path):
            visited.add(node)
            rec_stack.add(node)
            path.append(node)
            
            for neighbor in self.graph.get(node, set()):
                if neighbor not in visited:
                    if dfs(neighbor, path):
                        return True
                elif neighbor in rec_stack:
                    idx = path.index(neighbor)
                    cycle_path.extend(path[idx:])
                    return True
            
            path.pop()
            rec_stack.discard(node)
            return False
        
        for node in list(self.graph.keys()):
            if node not in visited:
                if dfs(node, []):
                    return True, cycle_path
        return False, []

# No deadlock case
print("  === Case 1: No Deadlock ===")
wfg1 = WaitForGraph()
wfg1.add_wait("T1", "T2")
wfg1.add_wait("T2", "T3")
has_cycle, path = wfg1.detect_cycle()
print(f"  Deadlock? {has_cycle}\n")

# Deadlock case
print("  === Case 2: Deadlock! ===")
wfg2 = WaitForGraph()
wfg2.add_wait("T1", "T2")
wfg2.add_wait("T2", "T3")
wfg2.add_wait("T3", "T1")  # Creates cycle!
has_cycle, path = wfg2.detect_cycle()
print(f"  Deadlock? {has_cycle}")
print(f"  Cycle: {' → '.join(path)} → {path[0]}")
print(f"  Victim (abort youngest): {path[-1]}")

In [ ]:
# ============================================
# 2. Deadlock Prevention: Wait-Die & Wound-Wait
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════╗
  ║       DEADLOCK PREVENTION PROTOCOLS                         ║
  ╠══════════════════════════════════════════════════════════════╣
  ║                                                              ║
  ║  WAIT-DIE (non-preemptive):                                 ║
  ║  • Older txn waits for younger. Younger txn dies (aborts).  ║
  ║  • "Old waits, young dies"                                   ║
  ║                                                              ║
  ║  WOUND-WAIT (preemptive):                                   ║
  ║  • Older txn wounds (aborts) younger. Younger txn waits.   ║
  ║  • "Old wounds, young waits"                                 ║
  ║                                                              ║
  ╚══════════════════════════════════════════════════════════════╝
""")

def wait_die(requester_ts, holder_ts):
    """Older waits, younger dies."""
    if requester_ts < holder_ts:  # older requests
        return "WAIT"
    else:
        return "DIE (abort & restart)"

def wound_wait(requester_ts, holder_ts):
    """Older wounds younger, younger waits."""
    if requester_ts < holder_ts:  # older requests
        return "WOUND (abort holder)"
    else:
        return "WAIT"

scenarios = [(1, 5), (5, 1)]  # (requester_timestamp, holder_timestamp)

for req, hold in scenarios:
    wd = wait_die(req, hold)
    ww = wound_wait(req, hold)
    label = "OLDER requests" if req < hold else "YOUNGER requests"
    print(f"  {label} (T{req} → T{hold}):")
    print(f"    Wait-Die:   {wd}")
    print(f"    Wound-Wait: {ww}\n")

In [ ]:
# ============================================
# 3. ARIES Recovery — Undo/Redo
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════╗
  ║            ARIES RECOVERY ALGORITHM                         ║
  ╠══════════════════════════════════════════════════════════════╣
  ║                                                              ║
  ║  Three Phases:                                               ║
  ║  1. ANALYSIS: Scan WAL to find dirty pages & active txns    ║
  ║  2. REDO:     Replay ALL logged actions (from checkpoint)   ║
  ║  3. UNDO:     Reverse uncommitted txn actions               ║
  ║                                                              ║
  ║  Timeline:                                                   ║
  ║  ──┬──────────┬──────────┬────────────┬──► time             ║
  ║    │checkpoint│  T1:W(A) │  T2:W(B)   │crash               ║
  ║    │          │  T1:COMMIT│            │                     ║
  ║                                                              ║
  ║  After crash:                                                ║
  ║  REDO: replay T1:W(A), T1:COMMIT, T2:W(B)                  ║
  ║  UNDO: T2 never committed → undo T2:W(B)                   ║
  ║                                                              ║
  ╚══════════════════════════════════════════════════════════════╝
""")

# Simulate ARIES
class ARIESRecovery:
    def __init__(self):
        self.wal = []
        self.data = {}
        self.committed = set()
    
    def log_and_apply(self, txn, key, old_val, new_val):
        self.wal.append({"txn": txn, "key": key, "old": old_val, "new": new_val, "type": "UPDATE"})
        self.data[key] = new_val
    
    def commit(self, txn):
        self.wal.append({"txn": txn, "type": "COMMIT"})
        self.committed.add(txn)
    
    def crash(self):
        print("  💥 CRASH! Data in memory lost.")
        self.data.clear()
    
    def recover(self):
        print("\n  === ARIES RECOVERY ===")
        
        # Phase 1: Analysis
        print("  Phase 1 (Analysis): Scanning WAL...")
        active_txns = set()
        for entry in self.wal:
            if entry["type"] == "UPDATE":
                active_txns.add(entry["txn"])
            elif entry["type"] == "COMMIT":
                active_txns.discard(entry["txn"])
        print(f"    Committed: {self.committed}")
        print(f"    Uncommitted (need undo): {active_txns}")
        
        # Phase 2: Redo ALL
        print("  Phase 2 (Redo): Replaying all updates...")
        for entry in self.wal:
            if entry["type"] == "UPDATE":
                self.data[entry["key"]] = entry["new"]
                print(f"    REDO: {entry['txn']} set {entry['key']}={entry['new']}")
        
        # Phase 3: Undo uncommitted
        print("  Phase 3 (Undo): Reversing uncommitted txns...")
        for entry in reversed(self.wal):
            if entry["type"] == "UPDATE" and entry["txn"] in active_txns:
                self.data[entry["key"]] = entry["old"]
                print(f"    UNDO: {entry['txn']} revert {entry['key']}={entry['old']}")
        
        print(f"\n  Recovered data: {self.data}")

db = ARIESRecovery()
db.log_and_apply("T1", "A", 100, 200)
db.commit("T1")
db.log_and_apply("T2", "B", 300, 400)  # Not committed!
print(f"  Before crash: {db.data}")
db.crash()
db.recover()
print("  → T1's change preserved (committed). T2's change undone (uncommitted).")

In [ ]:
# ============================================
# 4. Checkpoint optimization
# ============================================

print("""
  ┌──────────────────────────────────────────────────────────────────┐
  │ CHECKPOINTING — Limits Recovery Scan                            │
  ├──────────────────────────────────────────────────────────────────┤
  │                                                                  │
  │  Without checkpoint: scan ENTIRE WAL from beginning (slow!)     │
  │  With checkpoint: only scan from last checkpoint                 │
  │                                                                  │
  │  WAL: [op][op][op][CKPT][op][op][op][CRASH]                    │
  │                    ↑                                             │
  │                    Recovery starts here                          │
  │                                                                  │
  │  Fuzzy Checkpoint (most DBs):                                   │
  │  • Don't stop the world                                         │
  │  • Record active txns + dirty page list                         │
  │  • Flush dirty pages asynchronously                              │
  │                                                                  │
  │  Deadlock Handling in Practice:                                  │
  │  ┌──────────────────────────────────────────────────────┐       │
  │  │ MySQL: periodic wait-for graph check (every N sec)  │       │
  │  │ PostgreSQL: timeout-based (deadlock_timeout = 1s)   │       │
  │  │ Both: abort youngest/cheapest txn as victim         │       │
  │  └──────────────────────────────────────────────────────┘       │
  └──────────────────────────────────────────────────────────────────┘
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | How is deadlock detected? | Wait-for graph cycle detection (DFS). Abort victim (youngest/cheapest) |
| 2 | Wait-Die vs Wound-Wait? | Wait-Die: old waits, young dies. Wound-Wait: old wounds, young waits |
| 3 | What is ARIES? | Analysis→Redo→Undo. Redo ALL, then undo uncommitted. Industry standard |
| 4 | Why checkpoint? | Limits WAL scan on recovery. Fuzzy checkpoint: non-blocking |
| 5 | Redo vs Undo logging? | Redo: replay committed. Undo: reverse uncommitted. ARIES does both |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Deadlock = cycle in wait-for graph                   │
## │  • Prevention: Wait-Die/Wound-Wait (timestamp-based)    │
## │  • ARIES: Analysis → Redo all → Undo uncommitted       │
## │  • Checkpoint: limits recovery scope (fuzzy = no block) │
## │  • Victim selection: youngest or cheapest txn aborted   │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **B-Tree Indexing** — How indexes make queries fast